### 1. 使用dataprocessor对数据进行预处理操作

In [1]:
# %% 初始化环境
import sys
sys.path.append("..")
from experiments.environment import *
init_environment()
from pathlib import Path

Applied font.family: ['Times New Roman', 'Microsoft YaHei', 'DejaVu Sans']
findfont Times -> C:\Windows\Fonts\times.ttf
findfont CN -> C:\Windows\Fonts\msyh.ttc
🚀 环境初始化完成。


In [2]:
dataset_name = ['breast_cancer', 'diabetes_risk', 'framingham', 'hepatitis',
                'pima','tcga', 'thyroid']

train_ratio, val_ratio, test_ratio = 0.7, 0.1, 0.2
assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6 , "训练、验证和测试集比例之和必须为1"

In [4]:
import json
from collections import Counter
from sklearn.model_selection import train_test_split
from core.data_processor import TabularDataProcessor


for i in range(5):
    loc = Path('./splitted_data')
    splitted_loc = loc/f"seed_{i}"
    dataset_info_loc = Path('./data/dataset_info.json')

    with open(dataset_info_loc, 'r', encoding='utf-8') as f:
        info = json.load(f)
    
    for name in dataset_name:
        # 1. 读取数据集
        loc = './data/'+ name + '.csv'
        df_all = pd.read_csv(loc)

        # 2. 读取属性特性
        cat_cols = info[name]["categorical"]
        num_cols = info[name]["numerical"]
        target_col = info[name]["target"]

        # 2.5 定义预处理器
        processor = TabularDataProcessor(num_cols=num_cols, cat_cols=cat_cols, random_state=i + 1)

        '''unique_classes=df_all
        if Counter(cat_cols + num_cols) == Counter(df_all.columns[:-1]):
            print(f"{name}的属性完全匹配")
        else:
            print(f"{name}出问题了，检查一下json文件")
            set_all = set(cat_cols + num_cols)
            set_columns = set(list(df_all.columns)[:-1])
            missing = set_columns - set_all
            print(f"缺少的属性是：{missing}")'''
        
        # 3. 划分数据集
        unique_classes = df_all[target_col].nunique()
        stratify_col = df_all[target_col] if unique_classes > 1 else None

        df_temp, df_test = train_test_split(
                df_all,
                test_size=test_ratio,
                stratify=stratify_col,
                random_state=i,
            )
        
        stratify_col_temp = df_temp[target_col] if unique_classes > 1 else None 
        val_ratio_in_temp = val_ratio / (train_ratio + val_ratio)

        df_train, df_val = train_test_split(
            df_temp,
            test_size=val_ratio_in_temp,
            stratify=stratify_col_temp,
            random_state=i + 42
        )

        # 4. 缺失值补全
        processor.fit(df_train) # 使用训练集拟合
        real_train_imputed = processor.transform_readable(df_train)
        real_val_imputed = processor.transform_readable(df_val)
        real_test_imputed = processor.transform_readable(df_test)

        loc = splitted_loc / name
        loc.mkdir(parents=True, exist_ok=True)

        real_train_imputed.to_csv( loc/ "real_train_imputed.csv")
        real_val_imputed.to_csv(loc / "real_val_imputed.csv")
        real_test_imputed.to_csv(loc / "real_test_imputed.csv")        

In [5]:
pd.read_csv("../splitted_data/seed_0/pima/real_val_imputed.csv")

,Unnamed: 0,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,label
0,146,9,91,68,0,0,24.2,0.200,58,0
1,392,4,84,90,23,56,39.5,0.159,25,0
2,501,1,149,68,29,127,29.3,0.349,42,1
3,539,4,132,86,31,0,28.0,0.419,63,0
4,219,9,140,94,0,0,32.7,0.734,45,1
...,...,...,...,...,...,...,...,...,...,...
72,615,1,113,64,35,0,33.6,0.543,21,1
73,384,0,104,64,23,116,27.8,0.454,23,0
74,720,5,108,72,43,75,36.1,0.263,33,0
75,586,0,165,90,33,680,52.3,0.427,23,0
